In [1]:
import torch
import torchvision
from torchvision import datasets, transforms, models
from torch import nn, optim
from torch.utils.data import DataLoader, random_split, Dataset, WeightedRandomSampler
import numpy as np
import os

# Verify GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

# Paths
DATA_DIR = r"A:\AgriIntel_Project\AgriIntel\datasets\unified_dataset"
MODEL_SAVE_PATH = r"A:\AgriIntel_Project\AgriIntel\models\disease_model.pt"
CLASSES_SAVE_PATH = r"A:\AgriIntel_Project\AgriIntel\models\disease_classes.txt"

# Quick check — list classes and count images
class_names = sorted(os.listdir(DATA_DIR))
print(f"\nTotal classes: {len(class_names)}")

total_images = 0
for cls in class_names:
    cls_path = os.path.join(DATA_DIR, cls)
    count = len(os.listdir(cls_path))
    total_images += count
    print(f"  {cls}: {count}")

print(f"\nTotal images: {total_images}")

Using device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU

Total classes: 56
  Apple___Apple_scab: 723
  Apple___Black_rot: 621
  Apple___Cedar_apple_rust: 364
  Apple___healthy: 1736
  Blueberry___healthy: 1619
  Cherry_(including_sour)___Powdery_mildew: 1052
  Cherry_(including_sour)___healthy: 911
  Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: 581
  Corn_(maize)___Common_rust_: 1308
  Corn_(maize)___Northern_Leaf_Blight: 1177
  Corn_(maize)___healthy: 1162
  Grape___Black_rot: 1244
  Grape___Esca_(Black_Measles): 1383
  Grape___Leaf_blight_(Isariopsis_Leaf_Spot): 1076
  Grape___healthy: 492
  Orange___Haunglongbing_(Citrus_greening): 5507
  Peach___Bacterial_spot: 2297
  Peach___healthy: 472
  Pepper,_bell___Bacterial_spot: 1068
  Pepper,_bell___healthy: 1539
  Potato___Early_blight: 1117
  Potato___Late_blight: 1105
  Potato___healthy: 152
  Raspberry___healthy: 490
  Rice___Bacterial_Leaf_Blight: 2220
  Rice___Brown_Spot: 2246
  Rice___Healthy_Rice_Leaf: 653
  Rice___Leaf_

In [2]:
from torch.utils.data import Dataset

class SubsetWithTransform(Dataset):
    """Wrapper so train and val subsets get independent transforms."""
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        if self.transform:
            img = self.transform(img)
        return img, label


# Training transform — augmentation to help generalize
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Validation transform — no augmentation, just resize and normalize
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

print("Transforms defined.")
print(f"Train: {len(train_transform.transforms)} operations")
print(f"Val:   {len(val_transform.transforms)} operations")

Transforms defined.
Train: 8 operations
Val:   3 operations


In [3]:
# Load raw dataset WITHOUT any transform
full_dataset = datasets.ImageFolder(DATA_DIR)
class_names = full_dataset.classes
print(f"Classes found: {len(class_names)}")
print(f"Total images:  {len(full_dataset)}")

# Split 80% train, 20% val
train_size = int(0.8 * len(full_dataset))
val_size   = len(full_dataset) - train_size
train_raw, val_raw = random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Apply transforms independently using our fixed wrapper
train_dataset = SubsetWithTransform(train_raw, train_transform)
val_dataset   = SubsetWithTransform(val_raw,   val_transform)

print(f"\nTrain: {len(train_dataset)} images")
print(f"Val:   {len(val_dataset)} images")

# ── Weighted sampler to handle class imbalance ──────────────────────────────
# The problem: Wheat has ~128 images, Orange has 5,507 images.
# Without correction, the model sees Orange ~43x more often than Wheat
# and simply stops learning Wheat disease patterns.
# Solution: WeightedRandomSampler gives each class equal expected frequency.

# Count how many images of each class are in the TRAINING split
train_labels = [full_dataset.targets[i] for i in train_raw.indices]
class_counts = np.bincount(train_labels, minlength=len(class_names))
print(f"\nClass counts in training split:")
for i, (cls, count) in enumerate(zip(class_names, class_counts)):
    print(f"  {cls}: {count}")

# Weight for each class = 1 / count (rare classes get higher weight)
class_weights = 1.0 / (class_counts + 1e-6)  # +1e-6 avoids division by zero

# Each sample gets the weight of its class
sample_weights = [class_weights[label] for label in train_labels]
sample_weights = torch.DoubleTensor(sample_weights)

# Sampler draws samples with replacement, using those weights
# num_samples = len(train_dataset) means same number of batches per epoch
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_dataset),
    replacement=True
)

print(f"\nWeightedRandomSampler created.")
print(f"Highest weight class (rarest): {class_names[class_weights.argmax()]}")
print(f"Lowest weight class (most common): {class_names[class_weights.argmin()]}")

Classes found: 56
Total images:  70935

Train: 56748 images
Val:   14187 images

Class counts in training split:
  Apple___Apple_scab: 570
  Apple___Black_rot: 492
  Apple___Cedar_apple_rust: 289
  Apple___healthy: 1425
  Blueberry___healthy: 1294
  Cherry_(including_sour)___Powdery_mildew: 863
  Cherry_(including_sour)___healthy: 719
  Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: 451
  Corn_(maize)___Common_rust_: 1064
  Corn_(maize)___Northern_Leaf_Blight: 924
  Corn_(maize)___healthy: 913
  Grape___Black_rot: 994
  Grape___Esca_(Black_Measles): 1131
  Grape___Leaf_blight_(Isariopsis_Leaf_Spot): 852
  Grape___healthy: 403
  Orange___Haunglongbing_(Citrus_greening): 4420
  Peach___Bacterial_spot: 1849
  Peach___healthy: 377
  Pepper,_bell___Bacterial_spot: 856
  Pepper,_bell___healthy: 1266
  Potato___Early_blight: 882
  Potato___Late_blight: 874
  Potato___healthy: 113
  Raspberry___healthy: 400
  Rice___Bacterial_Leaf_Blight: 1754
  Rice___Brown_Spot: 1769
  Rice___Healthy_Ri

In [4]:
# DataLoaders
# Note: train_loader uses sampler instead of shuffle=True
# You cannot use both sampler and shuffle — sampler controls order instead
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=sampler,           # handles class balance
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f"Train batches per epoch: {len(train_loader)}")
print(f"Val batches per epoch:   {len(val_loader)}")

# Load pretrained ResNet18 — same architecture as before
# weights='IMAGENET1K_V1' means it starts with ImageNet knowledge
# We replace the final layer to match our 56 classes
model = models.resnet18(weights='IMAGENET1K_V1')
model.fc = nn.Linear(512, len(class_names))
model = model.to(device)

print(f"\nModel: ResNet18")
print(f"Output classes: {len(class_names)}")

# Loss, optimizer, scheduler — same as before
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

print("Ready to train.")

Train batches per epoch: 1774
Val batches per epoch:   444

Model: ResNet18
Output classes: 56
Ready to train.


In [5]:
NUM_EPOCHS = 7
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    # ── Training phase ──────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()

        if (batch_idx + 1) % 100 == 0:
            print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
                  f"Batch {batch_idx+1}/{len(train_loader)} | "
                  f"Loss: {train_loss/(batch_idx+1):.3f} | "
                  f"Train Acc: {100.*train_correct/train_total:.1f}%")

    # ── Validation phase ─────────────────────────────────────────────────────
    model.eval()
    val_correct = 0
    val_total = 0
    val_loss = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_acc   = 100. * val_correct / val_total
    train_acc = 100. * train_correct / train_total

    print(f"\n{'='*60}")
    print(f"EPOCH {epoch+1} COMPLETE")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Val Accuracy:   {val_acc:.2f}%")
    print(f"Val Loss:       {val_loss/len(val_loader):.3f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"New best model saved! Val Acc: {val_acc:.2f}%")

    scheduler.step()
    print(f"{'='*60}\n")

print(f"Training complete. Best val accuracy: {best_val_acc:.2f}%")

# Save class names — 56 classes in alphabetical order
with open(CLASSES_SAVE_PATH, 'w') as f:
    for cls in class_names:
        f.write(cls + '\n')
print(f"Class names saved to {CLASSES_SAVE_PATH}")

Epoch 1/7 | Batch 100/1774 | Loss: 2.057 | Train Acc: 43.2%
Epoch 1/7 | Batch 200/1774 | Loss: 1.662 | Train Acc: 52.8%
Epoch 1/7 | Batch 300/1774 | Loss: 1.466 | Train Acc: 57.4%
Epoch 1/7 | Batch 400/1774 | Loss: 1.312 | Train Acc: 61.8%
Epoch 1/7 | Batch 500/1774 | Loss: 1.201 | Train Acc: 64.7%
Epoch 1/7 | Batch 600/1774 | Loss: 1.132 | Train Acc: 66.6%
Epoch 1/7 | Batch 700/1774 | Loss: 1.073 | Train Acc: 68.1%
Epoch 1/7 | Batch 800/1774 | Loss: 1.025 | Train Acc: 69.3%
Epoch 1/7 | Batch 900/1774 | Loss: 0.982 | Train Acc: 70.6%
Epoch 1/7 | Batch 1000/1774 | Loss: 0.945 | Train Acc: 71.6%
Epoch 1/7 | Batch 1100/1774 | Loss: 0.911 | Train Acc: 72.6%
Epoch 1/7 | Batch 1200/1774 | Loss: 0.881 | Train Acc: 73.4%
Epoch 1/7 | Batch 1300/1774 | Loss: 0.854 | Train Acc: 74.1%
Epoch 1/7 | Batch 1400/1774 | Loss: 0.832 | Train Acc: 74.8%
Epoch 1/7 | Batch 1500/1774 | Loss: 0.813 | Train Acc: 75.3%
Epoch 1/7 | Batch 1600/1774 | Loss: 0.796 | Train Acc: 75.8%
Epoch 1/7 | Batch 1700/1774 | Los